In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import numpy as np
import networkx as nx
import random
from sklearn.metrics import accuracy_score

/home/codespace/.python/current/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = dgl.data.AmazonCoBuyComputerDataset()
g = dataset[0]
g

Graph(num_nodes=13752, num_edges=491722,
      ndata_schemes={'label': Scheme(shape=(), dtype=torch.int64), 'feat': Scheme(shape=(767,), dtype=torch.float32)}
      edata_schemes={'__orig__': Scheme(shape=(), dtype=torch.int64)})

# Q1

In [3]:
# (a)
num_nodes = g.number_of_nodes()

all_nodes = np.arange(num_nodes)

train_size = int(0.8 * num_nodes)

val_size = int(0.1 * num_nodes)

train_nodes = torch.tensor(all_nodes[:train_size])
val_nodes   = torch.tensor(all_nodes[train_size:train_size+val_size])
test_nodes  = torch.tensor(all_nodes[train_size+val_size:])

print("Train nodes:", len(train_nodes))
print("Validation nodes:", len(val_nodes))
print("Test nodes:", len(test_nodes))

Train nodes: 11001
Validation nodes: 1375
Test nodes: 1376


In [4]:
# (b)

# Original feature dimension
orig_feat_dim = g.ndata['feat'].shape[1]

# (i) All-One Features
features_all_one = torch.ones(g.num_nodes(), orig_feat_dim)

In [5]:
# (ii) Original Features
features_orig = g.ndata['feat']

In [6]:
# (iii) Graph structural features + one-hot vector
# Convert DGL graph to NetworkX for computing structural features
nx_g = g.to_networkx().to_undirected()

# Compute structural features using NetworkX
nx_g = nx.Graph(nx_g)
clustering_dict = nx.clustering(nx_g)
degree_dict = dict(nx_g.degree())
# Normalize degree values
max_degree = max(degree_dict.values())
degree_norm = {k: v/max_degree for k, v in degree_dict.items()}

# Betweenness centrality and eigenvector centrality 
betweenness_dict = nx.betweenness_centrality(nx_g)
eigenvector_dict = nx.eigenvector_centrality(nx_g, max_iter=1000)

# Create feature matrix from these dictionaries
structural_features = []
for i in range(g.num_nodes()):
    features_i = [
        clustering_dict[i],
        degree_norm[i],
        betweenness_dict[i],
        eigenvector_dict[i]
    ]
    structural_features.append(features_i)
structural_features = torch.tensor(structural_features, dtype=torch.float32)

# One-hot encoding for each node
one_hot = torch.eye(g.num_nodes(), dtype=torch.float32)

# Concatenate structural features with one-hot vectors
features_struct = torch.cat([structural_features, one_hot], dim=1)

In [7]:
from dgl.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_dim, num_classes):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_feats, hidden_dim, 'mean')
        self.conv2 = SAGEConv(hidden_dim, hidden_dim, 'mean')
        self.classify = nn.Linear(hidden_dim, num_classes)

    def forward(self, g, features):
        h = self.conv1(g, features)
        h = F.relu(h)
        h = self.conv2(g, h)
        h = F.relu(h)
        return self.classify(h)

# Assume the number of classes is given by the maximum label value +1
labels = g.ndata['label']
num_classes = labels.max().item() + 1

In [8]:
def train(model, g, features, labels, train_nodes, val_nodes, epochs=400, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        logits = model(g, features)
        loss = loss_fn(logits[train_nodes], labels[train_nodes])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch+1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                train_pred = logits[train_nodes].argmax(dim=1)
                train_acc = accuracy_score(labels[train_nodes].cpu(), train_pred.cpu())
                val_pred = logits[val_nodes].argmax(dim=1)
                val_acc = accuracy_score(labels[val_nodes].cpu(), val_pred.cpu())
            print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

    return model

def evaluate(model, g, features, labels, test_nodes):
    model.eval()
    with torch.no_grad():
        logits = model(g, features)
        test_pred = logits[test_nodes].argmax(dim=1)
        test_acc = accuracy_score(labels[test_nodes].cpu(), test_pred.cpu())
    print("Test Accuracy:", test_acc)
    return test_acc

In [9]:
print("Experiment (i): All-One Features")
model_allone = GraphSAGE(in_feats=features_all_one.shape[1], hidden_dim=64, num_classes=num_classes)
model_allone = train(model_allone, g, features_all_one, labels, train_nodes, val_nodes)
test_acc_allone = evaluate(model_allone, g, features_all_one, labels, test_nodes)

Experiment (i): All-One Features
Epoch 50: Loss = 1.6920, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 100: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 150: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 200: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 250: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 300: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 350: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 400: Loss = 1.6906, Train Acc = 0.4392, Val Acc = 0.0633
Test Accuracy: 0.08793604651162791


In [10]:
print("Experiment (ii): Original Features")
model_orig = GraphSAGE(in_feats=features_orig.shape[1], hidden_dim=64, num_classes=num_classes)
model_orig = train(model_orig, g, features_orig, labels, train_nodes, val_nodes)
test_acc_orig = evaluate(model_orig, g, features_orig, labels, test_nodes)

Experiment (ii): Original Features
Epoch 50: Loss = 0.6189, Train Acc = 0.8139, Val Acc = 0.6145
Epoch 100: Loss = 0.3374, Train Acc = 0.8942, Val Acc = 0.7571
Epoch 150: Loss = 0.2605, Train Acc = 0.9173, Val Acc = 0.8298
Epoch 200: Loss = 0.2176, Train Acc = 0.9312, Val Acc = 0.8625
Epoch 250: Loss = 0.1744, Train Acc = 0.9439, Val Acc = 0.8756
Epoch 300: Loss = 0.1394, Train Acc = 0.9562, Val Acc = 0.8960
Epoch 350: Loss = 0.4388, Train Acc = 0.8566, Val Acc = 0.6764
Epoch 400: Loss = 0.1996, Train Acc = 0.9335, Val Acc = 0.8415
Test Accuracy: 0.6984011627906976


In [11]:
print("Experiment (iii): Structural Features + One-Hot Vector")
model_struct = GraphSAGE(in_feats=features_struct.shape[1], hidden_dim=64, num_classes=num_classes)
model_struct = train(model_struct, g, features_struct, labels, train_nodes, val_nodes)
test_acc_struct = evaluate(model_struct, g, features_struct, labels, test_nodes)

Experiment (iii): Structural Features + One-Hot Vector
Epoch 50: Loss = 0.0002, Train Acc = 1.0000, Val Acc = 0.9062
Epoch 100: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.8996
Epoch 150: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.9018
Epoch 200: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.9025
Epoch 250: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.9025
Epoch 300: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.9018
Epoch 350: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.9018
Epoch 400: Loss = 0.0000, Train Acc = 1.0000, Val Acc = 0.9011
Test Accuracy: 0.5806686046511628


In [12]:
# (c)
# Define a deeper GraphSAGE model with 10 layers
class DeepGraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_dim, num_classes, num_layers=10):
        super(DeepGraphSAGE, self).__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_feats, hidden_dim, 'mean'))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim, 'mean'))
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, g, features):
        x = features
        for conv in self.convs:
            x = F.relu(conv(g, x))
        x = self.fc(x)
        return x

In [13]:
model_deep = DeepGraphSAGE(in_feats=features_orig.shape[1], hidden_dim=64, num_classes=num_classes)
model_deep = train(model_deep, g, features_orig, labels, train_nodes, val_nodes)
test_acc_deep = evaluate(model_deep, g, features_orig, labels, test_nodes)

Epoch 50: Loss = 1.6808, Train Acc = 0.4392, Val Acc = 0.0633
Epoch 100: Loss = 1.5934, Train Acc = 0.4839, Val Acc = 0.1309
Epoch 150: Loss = 1.5743, Train Acc = 0.4391, Val Acc = 0.0633
Epoch 200: Loss = 3.4040, Train Acc = 0.1754, Val Acc = 0.1084
Epoch 250: Loss = 1.2075, Train Acc = 0.5655, Val Acc = 0.2422
Epoch 300: Loss = 1.0825, Train Acc = 0.6074, Val Acc = 0.2291
Epoch 350: Loss = 0.9135, Train Acc = 0.6887, Val Acc = 0.5411
Epoch 400: Loss = 0.6780, Train Acc = 0.7723, Val Acc = 0.6393
Test Accuracy: 0.32049418604651164


The 2-layer GraphSAGE outperforms the 10-layer variant due to its ability to capture relevant local structure without over-smoothing, computational efficiency, and robustness in training.

# Q2

## (a)

In [14]:
# Function to sample negative edges (unconnected node pairs)
def sample_negative_edges(graph, num_samples):
    """Sample random node pairs that do not have an edge between them."""
    negative_edges = []
    while len(negative_edges) < num_samples:
        # Sample a large batch of random node pairs
        u = torch.randint(0, graph.num_nodes(), (num_samples * 10,))
        v = torch.randint(0, graph.num_nodes(), (num_samples * 10,))
        # Ensure u < v to represent undirected edges consistently
        mask = u < v
        u, v = u[mask], v[mask]
        # Check which pairs do not have an edge
        has_edge = graph.has_edges_between(u, v)
        non_edges = ~has_edge
        candidates = torch.stack([u[non_edges], v[non_edges]], dim=1)
        negative_edges.extend(candidates.tolist())
        if len(negative_edges) >= num_samples:
            break
    return torch.tensor(negative_edges[:num_samples])

In [15]:
# Step 1: Extract undirected edges (u < v) to avoid duplicates
u, v = g.edges()
mask = u < v  # Consider only one direction for undirected representation
undirected_edges = torch.stack([u[mask], v[mask]], dim=1)
num_undirected_edges = undirected_edges.shape[0]
print(f"Number of undirected edges: {num_undirected_edges}")

# Step 2: Split edges into training and testing sets (10% for testing)
test_size = int(0.1 * num_undirected_edges)
indices = torch.randperm(num_undirected_edges)
test_indices = indices[:test_size]
train_indices = indices[test_size:]

# Positive test samples (edges to be removed)
test_positive_samples = undirected_edges[test_indices]
# Positive train samples (remaining edges)
train_positive_samples = undirected_edges[train_indices]
print(f"Test positive samples: {test_positive_samples.shape[0]}")
print(f"Train positive samples: {train_positive_samples.shape[0]}")

Number of undirected edges: 245861
Test positive samples: 24586
Train positive samples: 221275


In [16]:
# Step 3: Remove test edges from the graph to create the training graph
# Since the graph is bidirected, remove both (u,v) and (v,u) for each test edge
test_eids = []
for u_node, v_node in test_positive_samples:
    eid_uv = g.edge_ids(u_node, v_node)
    eid_vu = g.edge_ids(v_node, u_node)
    test_eids.append(eid_uv)
    test_eids.append(eid_vu)
test_eids = torch.cat(test_eids)
train_graph = dgl.remove_edges(g, test_eids)
print(f"Training graph edges after removal: {train_graph.num_edges()}")

# Step 4: Generate negative test samples
test_negative_samples = sample_negative_edges(g, test_size)
print(f"Test negative samples: {test_negative_samples.shape[0]}")

# Step 5: Generate negative train samples, excluding those in the test set
train_negative_size = num_undirected_edges - test_size
# Sample more than needed initially to account for exclusions
train_negative_candidates = sample_negative_edges(g, train_negative_size * 2)
# Combine test positive and negative samples to exclude
test_samples = torch.cat([test_positive_samples, test_negative_samples], dim=0)
exclude_set = set(map(tuple, test_samples.tolist()))
# Filter out any candidates that are in the test set
train_negative_samples = [edge for edge in train_negative_candidates.tolist() 
                          if tuple(edge) not in exclude_set]

Training graph edges after removal: 442550
Test negative samples: 24586


In [17]:
# If not enough samples, generate additional ones
if len(train_negative_samples) < train_negative_size:
    additional = sample_negative_edges(g, train_negative_size - len(train_negative_samples))
    additional = [edge for edge in additional.tolist() if tuple(edge) not in exclude_set]
    train_negative_samples.extend(additional)
train_negative_samples = torch.tensor(train_negative_samples[:train_negative_size])
print(f"Train negative samples: {train_negative_samples.shape[0]}")

# Dataset is now ready for link prediction
print("\nDataset summary:")
print(f"Training graph: {train_graph.num_nodes()} nodes, {train_graph.num_edges()} edges")
print(f"Test positive samples shape: {test_positive_samples.shape}")
print(f"Test negative samples shape: {test_negative_samples.shape}")
print(f"Train positive samples shape: {train_positive_samples.shape}")
print(f"Train negative samples shape: {train_negative_samples.shape}")

Train negative samples: 221275

Dataset summary:
Training graph: 13752 nodes, 442550 edges
Test positive samples shape: torch.Size([24586, 2])
Test negative samples shape: torch.Size([24586, 2])
Train positive samples shape: torch.Size([221275, 2])
Train negative samples shape: torch.Size([221275, 2])


## (b)

In [18]:
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_feats, hidden_dim, 'mean')
        self.conv2 = SAGEConv(hidden_dim, hidden_dim, 'mean')

    def forward(self, g, features):
        h = self.conv1(g, features)
        h = F.relu(h)
        h = self.conv2(g, h)
        h = F.relu(h)
        return h

In [19]:
# Training setup
model = GraphSAGE(in_feats=features_struct.shape[1], hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

train_edges = torch.cat([train_positive_samples, train_negative_samples], dim=0)
train_labels = torch.cat([torch.ones(train_positive_samples.shape[0]), 
                          torch.zeros(train_negative_samples.shape[0])])

In [20]:
num_epochs = 400
for epoch in range(num_epochs):
    model.train()
    embeddings = model(train_graph, features_struct)
    u, v = train_edges[:, 0], train_edges[:, 1]
    pred = torch.sigmoid((embeddings[u] * embeddings[v]).sum(dim=1))
    loss = loss_fn(pred, train_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}: Loss = {loss.item():.4f}")

Epoch 50: Loss = 0.4422
Epoch 100: Loss = 0.4010
Epoch 150: Loss = 0.3883
Epoch 200: Loss = 0.3846
Epoch 250: Loss = 0.3824
Epoch 300: Loss = 0.3813
Epoch 350: Loss = 0.3805
Epoch 400: Loss = 0.3802


## (c)

In [21]:
from sklearn.metrics import roc_auc_score
# Function to compute edge scores (dot product of node embeddings)
def compute_edge_scores(embeddings, edges):
    """
    Compute edge scores as the dot product of node embeddings.
    
    Parameters:
    - embeddings: Tensor of shape [num_nodes, hidden_dim]
    - edges: Tensor of shape [num_edges, 2] with node pairs [u, v]
    
    Returns:
    - scores: Tensor of shape [num_edges]
    """
    u, v = edges[:, 0], edges[:, 1]
    scores = (embeddings[u] * embeddings[v]).sum(dim=1)
    return scores

## (d)

In [22]:
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
for epoch in range(400):
    model.train()  # Set model to training mode
    optimizer.zero_grad()  # Reset gradients for this epoch
    
    # Step 1: Generate node embeddings with the model
    embeddings = model(train_graph, features_struct)  # Shape: [num_nodes, hidden_dim]
    
    # Step 2: Prepare training data by combining positive and negative samples
    train_edges = torch.cat([train_positive_samples, train_negative_samples], dim=0)  # Shape: [num_train_edges, 2]
    train_labels = torch.cat([
        torch.ones(train_positive_samples.shape[0]),
        torch.zeros(train_negative_samples.shape[0])
    ], dim=0)  # Shape: [num_train_edges]
    
    # Step 3: Compute raw edge scores
    train_scores = compute_edge_scores(embeddings, train_edges)
    
    # Step 4: Apply sigmoid to convert raw scores into probabilities
    train_probs = torch.sigmoid(train_scores)
    
    # Step 5: Compute binary cross-entropy loss
    loss = F.binary_cross_entropy(train_probs, train_labels.to(train_probs.device))
    
    # Backpropagation and optimization step
    loss.backward()
    optimizer.step()
    
    # Print training progress and compute metrics
    if epoch % 50 == 0:
        model.eval()  # Switch to evaluation mode for metrics
        with torch.no_grad():
            # Evaluate on the training set
            train_auc = roc_auc_score(train_labels.cpu().numpy(), train_probs.cpu().numpy())
        print(f"Epoch {epoch}/{num_epochs} | Loss: {loss.item():.4f} | AUC: {train_auc:.4f}")


Epoch 0/400 | Loss: 0.3801 | AUC: 0.9769
Epoch 50/400 | Loss: 0.3859 | AUC: 0.9638
Epoch 100/400 | Loss: 0.3846 | AUC: 0.9647
Epoch 150/400 | Loss: 0.3839 | AUC: 0.9655
Epoch 200/400 | Loss: 0.3832 | AUC: 0.9663
Epoch 250/400 | Loss: 0.3830 | AUC: 0.9663
Epoch 300/400 | Loss: 0.3825 | AUC: 0.9668
Epoch 350/400 | Loss: 0.3822 | AUC: 0.9668


## (e)

In [23]:

# Generate node embeddings with the trained model
with torch.no_grad():  # No gradient computation needed for evaluation
    model.eval()  # Set model to evaluation mode
    embeddings = model(train_graph, features_struct)  # Shape: [num_nodes, hidden_dim]

# Prepare test data
test_edges = torch.cat([test_positive_samples, test_negative_samples], dim=0)  # Shape: [num_test_edges, 2]
test_labels = torch.cat([torch.ones(test_positive_samples.shape[0]), 
                         torch.zeros(test_negative_samples.shape[0])])  # Shape: [num_test_edges]

# Compute edge scores and probabilities for the test set
test_scores = compute_edge_scores(embeddings, test_edges)  # Raw dot product scores
test_probs = torch.sigmoid(test_scores).cpu().numpy()  # Probabilities (convert to numpy for sklearn)
test_labels = test_labels.cpu().numpy()  # Ground truth labels (convert to numpy)

# Compute AUC score
auc = roc_auc_score(test_labels, test_probs)
print(f"AUC Score on Test Set: {auc:.4f}")

AUC Score on Test Set: 0.9043
